# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)

d

# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
# Write your solution here
from pyspark.sql import functions as F

# Extracting 3 tables
bookings_df = spark.read.table("bookings")
facilities_df = spark.read.table("facilities")
members_df = spark.read.table("members")

# Transforming as per given conditions
result = bookings_df \
    .filter(
        (F.col("starttime") >= "2012-09-01") & 
        (F.col("starttime") < "2012-10-01")
    ) \
    .groupBy("facid") \
    .agg(F.sum("slots").alias("Total Slots")) \
    .orderBy("Total Slots")

#  Load: Save as a Parquet file into your Volume
# Defining path where I want to save it (volume)
volume_path = "/Volumes/workspace/default/etl_exercise/facility_slots_sept_2012.parquet"

result.write.mode("overwrite").parquet(volume_path)

print(f"File successfully saved to: {volume_path}")


File successfully saved to: /Volumes/workspace/default/etl_exercise/facility_slots_sept_2012.parquet


## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
# Write your solution here

from pyspark.sql import functions as F

# Extract
bookings = spark.read.table("bookings")
facilities = spark.read.table("facilities")
members = spark.read.table("members")

# Transform 
result = members.alias("mems") \
    .join(bookings.alias("bks"), F.col("mems.memid") == F.col("bks.memid")) \
    .join(facilities.alias("facs"), F.col("bks.facid") == F.col("facs.facid")) \
    .filter(F.col("facs.name").contains("Tennis Court")) \
    .select(
        F.concat_ws(" ", F.col("mems.firstname"), F.col("mems.surname")).alias("member"),
        F.col("facs.name").alias("facility")
    ) \
    .distinct() \
    .orderBy("member", "facility")

# Load: Partition by facility and save as a managed table
result.write \
    .partitionBy("facility") \
    .mode("overwrite") \
    .saveAsTable("threejoin_delta")

print("Job 2 Complete: Table 'threejoin_delta' created and partitioned.")

Job 2 Complete: Table 'threejoin_delta' created and partitioned.


In [0]:
# Display the data to ensure the join worked
df_verify = spark.read.table("threejoin_delta")
display(df_verify)

member,facility
Bader Florence,Tennis Court 1
Baker Anne,Tennis Court 1
Baker Timothy,Tennis Court 1
Boothe Tim,Tennis Court 1
Butters Gerald,Tennis Court 1
Coplin Joan,Tennis Court 1
Crumpet Erica,Tennis Court 1
Dare Nancy,Tennis Court 1
Farrell David,Tennis Court 1
Farrell Jemima,Tennis Court 1


In [0]:
# This SQL command shows how Spark has physically divided the data
display(spark.sql("SHOW PARTITIONS threejoin_delta"))

facility
Tennis Court 2
Tennis Court 1


## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
import requests
import time

url = "https://alpha-vantage.p.rapidapi.com/query"

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": 'cdbb794ca8msha0d0e0ca9a802e4p1d2170jsn5f7f62a7c64e'
}

symbols = {
    "Google": "GOOGL",
    "Apple": "AAPL",
    "Microsoft": "MSFT",
    "Tesla": "TSLA"
}

rows = []

for company, symbol in symbols.items():
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "outputsize": "compact",
        "datatype": "json"
    }

    retries = 5

    for attempt in range(retries):
        response = requests.get(url, headers=headers, params=params, timeout=30)

        print(f"{company} -> Attempt {attempt+1}: {response.status_code}")

        if response.status_code == 200:
            data = response.json()

            if "Time Series (Daily)" not in data:
                print(f"API error: {data}")
                break

            for dt, vals in data["Time Series (Daily)"].items():
                rows.append((
                    company,
                    symbol,
                    dt,
                    float(vals["1. open"]),
                    float(vals["2. high"]),
                    float(vals["3. low"]),
                    float(vals["4. close"]),
                    int(vals["5. volume"])
                ))

            break  # success

        elif response.status_code == 429:
            wait_time = 5 * (attempt + 1)  # ⏱exponential backoff
            print(f"⏱️ Rate limited. Waiting {wait_time}s...")
            time.sleep(wait_time)

        else:
            print(f"Error {response.status_code}: {response.text}")
            break

    # spacing between symbols
    time.sleep(12)  #(Alpha Vantage safe gap)

# Prevent Spark crash
if not rows:
    raise ValueError("No data fetched. Likely rate-limited or API issue.")

columns = ["company", "symbol", "date", "open", "high", "low", "close", "volume"]

stock_df = spark.createDataFrame(rows, columns)
stock_df = stock_df.withColumn("date", stock_df["date"].cast("date"))

stock_df.show(truncate=False)

Google -> Attempt 1: 200
Apple -> Attempt 1: 200
Microsoft -> Attempt 1: 200
Tesla -> Attempt 1: 200
+-------+------+----------+-------+--------+--------+------+--------+
|company|symbol|date      |open   |high    |low     |close |volume  |
+-------+------+----------+-------+--------+--------+------+--------+
|Google |GOOGL |2026-04-10|320.015|321.83  |316.32  |317.24|19152630|
|Google |GOOGL |2026-04-09|315.905|319.54  |311.06  |318.49|23739173|
|Google |GOOGL |2026-04-08|320.445|322.08  |315.02  |317.32|33547140|
|Google |GOOGL |2026-04-07|302.725|305.63  |297.72  |305.46|23205361|
|Google |GOOGL |2026-04-06|295.87 |300.62  |295.18  |299.99|16945494|
|Google |GOOGL |2026-04-02|290.69 |298.08  |289.45  |295.77|21666465|
|Google |GOOGL |2026-04-01|290.835|300.52  |290.41  |297.39|37684462|
|Google |GOOGL |2026-03-31|278.04 |288.08  |277.09  |287.56|43875400|
|Google |GOOGL |2026-03-30|276.42 |277.09  |272.11  |273.5 |35141244|
|Google |GOOGL |2026-03-27|277.275|279.37  |273.95  |274.34

## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
# Write your solution here
# EXTRACT 
# RNAcentral public database connection details
jdbc_url = "jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs"
connection_properties = {
    "user": "reader",
    "password": "NWDMCE5xdipIjRrp",
    "driver": "org.postgresql.Driver"
}

# limit of 100 records
sql_query = "(SELECT * FROM rna LIMIT 100) AS rna_subset"

rna_df = spark.read.jdbc(
    url=jdbc_url, 
    table=sql_query, 
    properties=connection_properties
)

# TRANSFORM
# No transformation required as per the instructions

# LOAD
# Save the results into a managed table
rna_df.write.mode("overwrite").saveAsTable("rna_100_records")

# Verify by displaying the results
display(spark.read.table("rna_100_records"))


id,upi,timestamp,userstamp,crc64,len,seq_short,seq_long,md5
20945691,URS00013F9B1B,2019-12-02T13:23:15.235Z,rnacen,B14B21EF07466D90,532,AAAACTAATGGGGAATCTTGCGCAATGGGGGAAACCCTGACGCAGCCATGCCGCGTGAGTGATGAAGGCCTTAGGGTTGTAAAACTCTTTCAGCGGGGACGATAATGACGGTACCCGCAGAAGAAGCCCCGGCTAACTCCGTGCCAGCAGCCGCGGTAATACGGAGGGGGCTAGCGTTGTTCGGAATTACTGGGCGTAAAGCGCGCGTAGGCGGCTTTGTAAGTTAGAGGTGAAAGCCCGGAGCTCAACTCCGGAATTGCCTTTAAGACTGCATCGCTAGAATCATGGAGAGGTTAGTGGAATTCCGAGTGTAGAGGTGAAATTCGTAGATATTCGGAAGAACACCAGTGGCGAAGGCGACTAACTGGACATGTATTGACGCTGAGGTACGAAAGCGTGGGGAGCAAACAGGATTAGATACCCTGGTAGTCCACGCCGTAAACGATGATGACTAGCTGTCGGGGCGCTTAGCGTTTCGGTGGAGCAGCTAACGCGTTAAGTCATCCGCCTGGGGAGTACGGCCGCAAGGTTA,null,7bbcdbbc3601fe9e56b121c432c5f255
20945692,URS00013F9B1C,2019-12-02T13:23:15.235Z,rnacen,59C495343ACD45D2,465,CCTACGGGGGGCAGCAGTAGGGAATCTTCGGCAATGGACGAAAGTCTGACCGAGCAACGCCGCGTGAGTGAAGAAGGTTTTCGGATCGTAAAACTCTGTTGTTAGAGAAGAACGTTGCATAGAGTGGAAAATTATGCAAGTGACGGGATCTAACCAGAAAGGGACGGCTAACTACGTGCCAGCAGCGGCGGTAAAACGTAGGTACCGAGCGTTGTCCGGATTTATTGGGCGTAAAGCGAGCGCAGGTGGTTTAATAAGTCTGATGTAAAAGGCAGTGGCTCAACCATTGTGTGCATTGGAAACTGTTAGACTTGAGTGCAGTAGAGGAGAGTGGAATTCCATGTGTAGCGGTGAAATGCGTAGATATATGGAGGAACACCGGTGGCGAAAGCGGCTCTCTGGACTGTCACTGACACTGAGGCTCGAAAGCGTGGGTAGCAAACAGGATTAGATACCCCTGTAGTC,null,7bbcdc1d11f4b8c67a605954f989d976
20945693,URS00013F9B1D,2019-12-02T13:23:15.235Z,rnacen,B628713A9E8184CF,220,TACGGAGGGTGCAAGCGTTGTTCGGATTTACTGGGCGTAAAGCGCGCGCAGGTGGATGGACAAGTCGGAAGTGAAATCTTGAGGCTCAACCTCGAGGCTGCTCCCGAAACTGTTCGTCTAGGGACTGGTAGAGGCTGGTAGAATTCCCGGTGTAGCGGTGAAATGCGTAGATATCGGGGAAGACCGACGAGCGCAGGACGGAAGTCACCTGGGGTTGTTC,null,7bbcdd71e70923fcb573bdbce705886d
20945694,URS00013F9B1E,2019-12-02T13:23:15.235Z,rnacen,CE057192092688A4,363,GCTTAGCATATATTATTGATGTTGCAGTTAAAACGTTCGTAGTCGACTCCTATTTACTAATATATGCAAAACTATGGGTTTTGTCATATTGTCAGTAAATGCAATATAGGTACAATATGTGTTTCATATATTGTGACTTATATCTTACTTTGAGTGAATCGGCGTGTTCCACGCAGGCATAAGCCGTGAACGATTTACCATAGAATAAATATACTATAACAATATTTTTATGTCTGTTCGGAAATATTGTTTATAATAATAAGAACGGTTGAGGGCATTCATATTCAACTGCTAGAGGTGAAATTCTTTGATTAGTTGAGGATGAACAATGGCGAAAGCATCTGCCAAGGACGTTTCTGTTGA,null,7bbcdf8a9a6094fd40aea965dc652943
20945695,URS00013F9B1F,2019-12-02T13:23:15.235Z,rnacen,403F449167FF72F8,459,CCTACGGGGGCAGCAGTGAGGAATATTGGTCAATGGACGAGAGTCTGAACCAGCCAAGTTGCGTGAAGGATGACTGCCCTAAGGGTTGTAAACTTCTTTTATATGGGAATAAAATGTTTCACGTGTGGGATTTTGTTTGTACCATATTAATAAGGATTGGGTAACTTCGTTGCAGGAGCCGCGGTAATACGGAGGATCCGAGCGTTATCCGGATTTTTTGGGTTTAAAGGGAGCGTAGGTGGATTGTTAAGTCAGTTGTGAAAGTTTGCGGATCAACCGTAAAATTGCAGTTGAAACTGGCAGTCTTGAGTACAGTAGAGGTGGGCGGAATTCGTGGTGTAGCGGTGAAATGCTTAGATATCACGAAGAACTCCGATTGCGAAGGCAGCTCACTGGACTGCAACTGACACTGAGGCTCGAAAGTGTGGGTATCAAACAGGATTAGATACCCTTGTAGTC,null,7bbce38a75845efabfd977c561ea4465
20945696,URS00013F9B20,2019-12-02T13:23:15.235Z,rnacen,899CEDAB1E84B3F1,254,GACGTAGGGCGCGAGCGTTGTTCGGATTTACTGGGCGTAAAGGGCGCGTAGGCGGCGCGGTAAGTCACTTGTGAAATCTCTGAGCTTAACTCAGAACGGCCAAGTGATACTGCAGTGCTAGAGTGCAGAAAGGGCAATCGGAATTCTTGGTGTAGCGGTGAAATGCGTAGATATCAAGAGGAACACCTGAGGCGAAGGCGGGTTGCTAGGCTGACACTGACGCTGAGTGCGCGAAAGCCAGGGGAGCAAACGGG,null,7bbce648ec9dddf6f2cee0907768a3cd
20945697,URS00013F9B21,2019-12-02T13:23:15.235Z,rnacen,FF5810410F12E024,440,CCTACGGGAGGCTGCAGTGGGGGATATTGCACAATGGGGGAAACCCTGATGCAGCGACGCCGCGTGTGGGAAGACGGTCTTCTGGATTGTAAACCACTGTCCCCAGGGACGAAAATGACGGTACCTGGGGAGGAAGCTCCGGCTAACTACGTGCCAGCAGCCGCGGTAATACGTAGGGAGCAAGCGTTGTCCGGAATTACTGGGTGTAAAGGGAGCGTAGGCGGGGATGCAAGTTGGGTGTCAAAACTACGGGCTCAACCGATAGTCGCACTCAAAACTGCAACTCTTGAGTGAAGTAGAGGCAGGCGGAATTCCTAGTGTAGCGGTGAAATGCGTAGATATTAGGAGGAACACCAGTGGCGAAGGCGGCCTGCTGGGCTTTTACTGACGCTGAGGCTCGAAAGTGTGGGGAGCAAACAGGATTAGATACCCTAGTAGTC,null,7bbce679b1841e4cb6c2d5e688db2013
20945698,URS00013F9B22,2019-12-02T13:23:15.235Z,rnacen,C082F17BA3DEAEC0,559,AAAACCGATAGGGAATCTTCGGCAATGGACGAAAGTCTGACCGAGCAACGCCGCGTGAGTGATGAAGGTTTTCGGATCGTAAAACTCTGTTGTTAGAGAAGAACAAGTACCGTTTGAATAAGGCGGTACCTTGACGGTACCTAACGAGAAAGCCCCGGCTAACTACGTGCCAGCAGCCGCGGTAATACGTAGGGGGCAAGCGTTGTCCGGAATTATTGGGCGTAAAGCGCGCGCAGGCGGTCTCTTAAGTCTGATGTGAAAGCCTACGGCTCAACCGTGGAGGGTCATTGGAAACTGGGGGACTTGAGGGTAGGAGAGGAAAGTGG